# Research 3 — official Qwen3.8-27B readiness only

This notebook does **not** load, train, or run a model. No Drive mount. Preserve the earlier community GGUF notebook. Use only the authorized 50 compute units; no purchases. Complete account/terms review first.

Use CPU initially for model-free probes to avoid unnecessary GPU billing. Inspect Settings → Subscription and runtime resources for actual balance and GPU unit rate. Prefer A100 for the later pilot; never start full 27B training on T4. A selected GPU is not evidence of availability.


In [ ]:
# Researcher-entered observations from Colab UI, not assumptions.
OBSERVED_START_BALANCE = None
OBSERVED_CURRENT_BALANCE = None
OBSERVED_RATE_UNITS_PER_HOUR = None
print("Authorized ceiling: 50 units; actual balance/rate still require UI verification.")


In [ ]:
"""Trusted, model-free Colab inventory. This is not a worker isolation gate."""
import json
import os
import platform
import shutil
import subprocess
from datetime import datetime, timezone
from pathlib import Path


def command(args):
    try:
        result = subprocess.run(args, capture_output=True, text=True, timeout=10)
        return {"exit_code": result.returncode, "stdout": result.stdout[:8192],
                "stderr": result.stderr[:2048]}
    except (OSError, subprocess.TimeoutExpired) as error:
        return {"error": type(error).__name__}


def inventory():
    memory = {}
    if Path('/proc/meminfo').exists():
        for line in Path('/proc/meminfo').read_text().splitlines():
            key, value = line.split(':', 1)
            if key in ('MemTotal', 'MemAvailable', 'SwapTotal'):
                memory[key + '_kib'] = int(value.strip().split()[0])
    root = '/content' if Path('/content').exists() else '.'
    gpu = command(['nvidia-smi', '--query-gpu=name,memory.total,memory.free',
                   '--format=csv,noheader'])
    # Probe namespace availability only. No network or host configuration changes.
    namespace = command(['unshare', '--user', '--map-root-user', '--net',
                         '--pid', '--fork', '/usr/bin/true'])
    mounts = Path('/proc/self/mountinfo').read_text() if Path('/proc/self/mountinfo').exists() else ''
    return {
        'utc': datetime.now(timezone.utc).isoformat(),
        'python': platform.python_version(), 'platform': platform.platform(),
        'uid': os.getuid() if hasattr(os, 'getuid') else None,
        'cpu_count': os.cpu_count(), 'memory': memory,
        'disk_free_bytes': shutil.disk_usage(root).free,
        'gpu': gpu, 'namespace_availability_probe': namespace,
        'bubblewrap_path': shutil.which('bwrap'),
        'drive_mount_lines': [line for line in mounts.splitlines()
                              if '/content/drive' in line or 'drivefs' in line],
        # Report names only, never secret values. Parent may be authenticated;
        # worker must have a separate clean environment and restricted filesystem.
        'sensitive_environment_variable_names': sorted(
            name for name in os.environ
            if any(part in name.upper() for part in ('TOKEN', 'SECRET', 'PASSWORD', 'CREDENTIAL'))),
        'isolation_verified': False,
        'model_worker_started': False,
        'notes': ['Namespace availability does not establish credential, filesystem, '
                  'network, CUDA, seccomp, cgroup, or watchdog containment. '
                  'Agentic evaluation remains prohibited until independent worker probes pass.'],
    }



report = inventory()
print(json.dumps(report, indent=2))


In [ ]:
from datetime import datetime, timezone
import uuid
RUN_ID = "qwen38_readiness_" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "_" + uuid.uuid4().hex[:8]
run_root = Path("/content/sp_lense_work") / RUN_ID
if report["drive_mount_lines"]:
    raise RuntimeError("Drive is mounted: no worker may start. Inventory only.")
run_root.mkdir(parents=True, exist_ok=False)
for relative in ("checkpoints/adapters", "checkpoints/resume", "training/configs", "training/logs", "training/receipts", "evaluation/scenarios", "evaluation/trajectories", "evaluation/tool_calls", "evaluation/results", "activations", "reports"):
    (run_root / relative).mkdir(parents=True, exist_ok=False)
manifest = {"run_id": RUN_ID, "status": "readiness_only", "authorized_compute_units": 50, "observed_start_balance": OBSERVED_START_BALANCE, "observed_current_balance": OBSERVED_CURRENT_BALANCE, "observed_rate_units_per_hour": OBSERVED_RATE_UNITS_PER_HOUR, "model": "Qwen/Qwen3.8-27B", "revision": "1d4bf0f2ff6012fd82039f2fa52739d0dd7c60c0", "worker_started": False, "isolation_verified": False, "inventory": report}
(run_root / "manifest.json").write_text(json.dumps(manifest, indent=2))
print("Local readiness receipt:", run_root / "manifest.json")


## Stop here

The restricted GPU worker, independent forbidden-operation/resource tests, software compatibility pilot, save/reload/resume test, baseline gate, scientific freeze and budget-admitted training are subsequent stages. Successful inventory alone authorizes none of them. The main experiment must not use a community GGUF or old 0.8B adapter. Never interpret namespace availability or offline library flags as verified containment.
